# 00 — Setup & Dataset
### Python Learning Guide · Colab Edition · Sales 2026
---
**Objective:** mount Google Drive, generate the Sales 2026 dataset into `MyDrive/sales2026/data/`, and verify every reference figure the guide quotes.

**Run this once before Module 1.1.** After that, re-run it any time you need the data back.

| # | Output | Rows | Notes |
|---|--------|------|-------|
| 1 | `data/products.csv` | 8 | Catalogue: code, category, price, stock, reorder point |
| 2 | `data/customers.csv` | 37 | Accounts by region and segment — 3 of them dormant |
| 3 | `data/sales.csv` | 652 | 646 clean order lines + 6 deliberately broken rows |

**Expected runtime:** under a minute, most of it the Drive permissions dialog.

## 1 · Install

Colab already ships pandas, numpy and requests. These are the extras the later modules need. Re-run this cell in any notebook that needs them — installs do not survive a runtime restart.

In [ ]:
# Cell 1 — packages the guide needs beyond Colab's defaults
!pip install -q "fastapi[standard]" pytest pytest-cov mypy

import sys, pandas
print(sys.version.split()[0], "· pandas", pandas.__version__)
print("If a later import fails: Runtime -> Restart session, then re-run from here.")

## 2 · Mount Drive & set paths

The first run opens a Google permissions dialog. Grant access to Drive — Colab needs it to see your files.

`ROOT`, `DATA`, `OUT`, `SRC` live in Drive and survive everything. `BIG` lives on the runtime's local disk and is wiped whenever the session ends, which is intentional.

In [ ]:
# Cell 2 — mount Drive and lay out the project.
# This is the standard paths cell. Every notebook in this guide starts with it.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/sales2026")   # project lives in Drive
DATA = ROOT / "data"                              # the three CSVs
OUT  = ROOT / "outputs"                           # reports you generate
SRC  = ROOT / "src"                               # .py modules (Phase 3+)
BIG  = Path("/content/data_big")                  # scratch — dies with the runtime

for d in (ROOT, DATA, OUT, SRC, BIG):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("CSVs:", sorted(p.name for p in DATA.glob("*.csv")) or "none yet")

## 3 · Save the generator

`%%writefile` writes the cell to a file instead of running it. It must be the first line of the cell, and it takes a literal path — no f-strings, no variables.

This is also the last time you will paste a long script into a cell. From Phase 3 onward this is how all your real code gets written.

In [ ]:
%%writefile /content/drive/MyDrive/sales2026/make_dataset.py
"""make_dataset.py — generate the Sales 2026 dataset used by every exercise.

Creates three CSVs in ./data:
    products.csv    8 rows   dimension: catalogue, price, stock
    customers.csv  37 rows   dimension: account, region, segment (3 dormant)
    sales.csv     652 rows   fact: one row per order line (646 clean + 6 deliberately bad)

Deterministic: same seed, same files, every time.
    python make_dataset.py                # 646 clean order lines
    python make_dataset.py --rows 412908  # big file for the streaming exercises
"""

from __future__ import annotations

import argparse
import csv
import random
from datetime import date, timedelta
from pathlib import Path

SEED = 2026
START = date(2026, 1, 1)
END = date(2026, 6, 30)

# product_code, product, category, unit_price, stock, reorder_point
PRODUCTS = [
    ("PS-100", "Pump seal",         "Rotating equipment",  1450.00, 240,  60),
    ("BR-500", "Bearing housing",   "Rotating equipment",  6750.00,  48,  20),
    ("IM-600", "Impeller",          "Rotating equipment", 12300.00,   8,  12),
    ("VK-200", "Valve kit",         "Valves & fittings",   8900.00,  62,  25),
    ("GS-300", "Gasket set",        "Valves & fittings",    320.00, 900, 200),
    ("CV-800", "Check valve",       "Valves & fittings",   4150.00,   9,  15),
    ("FM-400", "Flow meter",        "Instrumentation",    24500.00,   5,   8),
    ("PG-700", "Pressure gauge",    "Instrumentation",     1980.00, 130,  40),
]

SEGMENTS = ["Manufacturing", "Utilities", "Marine", "Construction"]

# region -> (number of active accounts, number of clean order lines)
REGIONS = {"NCR": (18, 406), "Cebu": (10, 142), "Davao": (6, 98)}

# Accounts on the books that never placed an order in the period. They exist so
# the anti-join exercises ("which customers bought nothing?") return real rows.
DORMANT = [
    ("C-NCR-019", "NCR Industrial Account 19",   "NCR",   "Construction"),
    ("C-CEB-011", "Cebu Industrial Account 11",  "Cebu",  "Utilities"),
    ("C-DAV-007", "Davao Industrial Account 07", "Davao", "Marine"),
]

# Regional demand mix. NCR skews to rotating equipment, Cebu to valves,
# Davao is mixed — this is what makes the GROUP BY exercises interesting.
MIX = {
    "NCR":   [0.20, 0.14, 0.10, 0.12, 0.18, 0.08, 0.06, 0.12],
    "Cebu":  [0.10, 0.06, 0.04, 0.26, 0.28, 0.14, 0.04, 0.08],
    "Davao": [0.22, 0.10, 0.06, 0.14, 0.20, 0.10, 0.08, 0.10],
}

# Six deliberately broken rows. Module 2.2 logs and skips them;
# Module 2.3 drops the duplicate. Do not "fix" them in the generator.
BAD_ROWS = [
    ("PH-01412", "2026-03-11", "C-NCR-004", "PS-100",  "NCR",   ""),      # blank qty
    ("PH-01488", "2026-03-29", "C-CEB-002", "GS-300",  "Cebu",  "six"),   # qty not numeric
    ("PH-01041", "2026-01-04", "C-NCR-001", "PS-100",  "NCR",   "12"),    # duplicate order_id
    ("PH-01530", "2026-04-14", "C-DAV-003", "ZZ-999",  "Davao", "4"),     # unknown product
    ("PH-01602", "",           "C-NCR-011", "VK-200",  "NCR",   "3"),     # missing date
    ("PH-01655", "2026-05-30", "C-CEB-007", "FM-400",  "Cebu",  "-2"),    # negative qty
]


def build_customers() -> list[dict]:
    rng = random.Random(SEED)
    rows = []
    for region, (n_accounts, _) in REGIONS.items():
        prefix = {"NCR": "NCR", "Cebu": "CEB", "Davao": "DAV"}[region]
        for i in range(1, n_accounts + 1):
            rows.append({
                "customer_id": f"C-{prefix}-{i:03d}",
                "customer": f"{region} Industrial Account {i:02d}",
                "region": region,
                "segment": rng.choice(SEGMENTS),
            })
    fields = ["customer_id", "customer", "region", "segment"]
    rows += [dict(zip(fields, d)) for d in DORMANT]
    return rows


def build_sales(customers: list[dict], scale: int | None) -> list[dict]:
    rng = random.Random(SEED + 1)
    dormant = {d[0] for d in DORMANT}
    by_region: dict[str, list[str]] = {r: [] for r in REGIONS}
    for c in customers:
        if c["customer_id"] not in dormant:
            by_region[c["region"]].append(c["customer_id"])

    codes = [p[0] for p in PRODUCTS]
    span = (END - START).days

    # Scale each region's line count proportionally when --rows is given.
    base_total = sum(n for _, n in REGIONS.values())
    counts = {r: n for r, (_, n) in REGIONS.items()}
    if scale:
        counts = {r: round(n * scale / base_total) for r, n in counts.items()}

    rows, oid = [], 1041
    for region, n_lines in counts.items():
        for _ in range(n_lines):
            code = rng.choices(codes, weights=MIX[region])[0]
            price = dict((p[0], p[3]) for p in PRODUCTS)[code]
            # Cheap consumables move in bulk; capital items move in ones and twos.
            qty = rng.randint(10, 60) if price < 1000 else (
                rng.randint(3, 18) if price < 5000 else rng.randint(1, 5))
            rows.append({
                "order_id": f"PH-{oid:05d}",
                "order_date": (START + timedelta(days=rng.randint(0, span))).isoformat(),
                "customer_id": rng.choice(by_region[region]),
                "product_code": code,
                "region": region,
                "qty": str(qty),
            })
            oid += 1

    rows.sort(key=lambda r: r["order_date"])
    if not scale:  # only pollute the small teaching file
        fields = ["order_id", "order_date", "customer_id", "product_code", "region", "qty"]
        rows += [dict(zip(fields, bad)) for bad in BAD_ROWS]
    return rows


def write_csv(path: Path, rows: list[dict], fields: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)
    print(f"wrote {path}  ({len(rows)} rows)")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--rows", type=int, default=None,
                    help="total clean order lines (default 646)")
    ap.add_argument("--out", type=Path, default=Path("data"))
    args = ap.parse_args()

    products = [dict(zip(
        ["product_code", "product", "category", "unit_price", "stock", "reorder_point"],
        (row[0], row[1], row[2], f"{row[3]:.2f}", row[4], row[5]),
    )) for row in PRODUCTS]
    customers = build_customers()
    sales = build_sales(customers, args.rows)

    write_csv(args.out / "products.csv", products,
              ["product_code", "product", "category", "unit_price", "stock", "reorder_point"])
    write_csv(args.out / "customers.csv", customers,
              ["customer_id", "customer", "region", "segment"])
    write_csv(args.out / "sales.csv", sales,
              ["order_id", "order_date", "customer_id", "product_code", "region", "qty"])


if __name__ == "__main__":
    main()

## 4 · Generate the data

`%cd` (magic) persists across cells; `!cd` (shell) does not. The generator writes to `./data`, so we change directory first.

Regenerating is always safe — it overwrites, never appends.

In [ ]:
%cd /content/drive/MyDrive/sales2026
!python make_dataset.py
!wc -l data/*.csv

Expected line counts (header included): **9**, **38**, **653**.

## 5 · Look at it

Turn on Colab's interactive table renderer once — sortable, filterable, paginated. Then read the three files.

In [ ]:
%load_ext google.colab.data_table
import pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

products  = pd.read_csv(DATA / "products.csv")
customers = pd.read_csv(DATA / "customers.csv")
sales     = pd.read_csv(DATA / "sales.csv")

print("products ", products.shape)
print("customers", customers.shape)
print("sales    ", sales.shape)
products

### The first thing to notice

`sales.dtypes` shows `qty` as **object**, not int. One row contains the word `"six"`, and one value is enough to make the whole column text.

That is not a bug in the file — it is the file doing its job. Six rows at the bottom of `sales.csv` are broken on purpose so Modules 2.1–2.3 have something real to defend against.

In [ ]:
print(sales.dtypes, "\n")
print("last 7 rows — the deliberate mess:")
sales.tail(7)

## 6 · Clean it

Five rows fail a row-level check. The sixth — a duplicate `order_id` — passes every field check and is only caught by a uniqueness constraint.

**652 − 5 − 1 = 646.**

In [ ]:
sales["qty"] = pd.to_numeric(sales["qty"], errors="coerce")
sales["order_date"] = pd.to_datetime(sales["order_date"], errors="coerce")

valid = (
    sales["qty"].notna()
    & (sales["qty"] > 0)
    & sales["order_date"].notna()
    & sales["product_code"].isin(products["product_code"])
)
clean = sales[valid].drop_duplicates(subset=["order_id"])

print(f"raw rows      {len(sales)}")
print(f"failed checks {(~valid).sum()}")
print(f"duplicates    {valid.sum() - len(clean)}")
print(f"clean rows    {len(clean)}   <- must be 646")

## 7 · Join and derive revenue

There is no `revenue` column, and that is deliberate. Revenue is `qty × unit_price`, and `unit_price` lives in a different file — so every meaningful question requires a join.

In Excel that is `XLOOKUP`. In SQL it is `JOIN … ON`. Here it is `.merge()`. Same move, three syntaxes.

Note the column selection on `customers`: it also has a `region` column, and without the selection pandas hands you `region_x` and `region_y`.

In [ ]:
df = (clean
      .merge(products, on="product_code", how="left")
      .merge(customers[["customer_id", "customer", "segment"]], on="customer_id", how="left"))
df["revenue"] = df["qty"] * df["unit_price"]

print(df.shape)
df[["order_id", "order_date", "product", "region", "qty", "unit_price", "revenue"]].head()

## 8 · Verify every reference figure

This is the answer key. If any assertion fails, the data is not what the guide describes — regenerate from Section 4 rather than debugging your analysis.

In [ ]:
import math

by_region = (df.groupby("region")
               .agg(revenue=("revenue", "sum"), orders=("order_id", "nunique"))
               .sort_values("revenue", ascending=False))
by_category = (df.groupby("category")
                 .agg(revenue=("revenue", "sum"), orders=("order_id", "nunique"))
                 .sort_values("revenue", ascending=False))

expected_region = {"NCR": (10_519_870.0, 406),
                   "Cebu": (3_514_180.0, 142),
                   "Davao": (2_300_500.0, 98)}

checks = []
for region, (rev, orders) in expected_region.items():
    got = by_region.loc[region]
    checks.append((f"{region} revenue", got.revenue, rev))
    checks.append((f"{region} orders",  got.orders,  orders))

checks += [
    ("clean rows",        len(clean),                                        646),
    ("total revenue",     df.revenue.sum(),                                  16_334_550.0),
    ("inventory value",   (products.stock * products.unit_price).sum(),      2_027_450.0),
    ("below reorder pt",  int((products.stock < products.reorder_point).sum()), 3),
    ("dormant accounts",  len(set(customers.customer_id) - set(clean.customer_id)), 3),
    ("distinct products", products.product_code.nunique(),                   8),
    ("top product rev",   df.groupby("product").revenue.sum().max(),         2_901_400.0),
]

ok = True
for name, got, want in checks:
    passed = math.isclose(float(got), float(want), rel_tol=1e-9)
    ok &= passed
    print(f"{'PASS' if passed else 'FAIL'}  {name:<18} got {got:>14,.2f}  want {want:>14,.2f}")

print("\nALL CHECKS PASSED" if ok else "\nSOMETHING IS OFF — regenerate the dataset")
display(by_region)
display(by_category)

## 9 · Save a summary to Drive

Anything you want tomorrow goes in Drive. `OUT` is where reports belong — this file is still there after the runtime dies.

In [ ]:
import json

by_region.to_csv(OUT / "region_summary.csv")

summary = {
    "clean_rows": int(len(clean)),
    "rejected_rows": int(len(sales) - len(clean)),
    "total_revenue": float(df.revenue.sum()),
    "by_region": {r: float(v) for r, v in by_region.revenue.items()},
    "by_category": {c_: float(v) for c_, v in by_category.revenue.items()},
    "date_range": [str(df.order_date.min().date()), str(df.order_date.max().date())],
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
!ls -la "{OUT}"

## 10 · The big file (Module 3.2 only)

Skip this until you reach Module 3.2. It writes to `/content`, **not** Drive — streaming 412,908 rows over the Drive mount is roughly ten times slower, and the delay is network latency rather than Python, which would hide the exact thing that module is trying to show you.

It disappears when the runtime recycles. Re-running this cell is the fix.

In [ ]:
%cd /content/drive/MyDrive/sales2026
!python make_dataset.py --rows 412908 --out /content/data_big
!ls -lh /content/data_big/sales.csv

## Next steps

1. **Prove it is reproducible.** *Runtime → Restart and run all.* The whole notebook must run top-to-bottom with no manual steps. If it does not, you have hidden state — that is the single most common way notebook analyses go wrong, and it is invisible in the saved file.
2. **Start Module 1.1** in a new notebook, `11_setup_colab.ipynb`. Use `_module_template.ipynb` as the starting point.
3. **Keep this notebook.** It is your dataset reset button — delete `data/` and re-run whenever you want a clean start.

| Path | Survives restart? | Use for |
|------|-------------------|---------|
| `MyDrive/sales2026/data/` | yes | The three CSVs |
| `MyDrive/sales2026/outputs/` | yes | Reports, logs, exports |
| `MyDrive/sales2026/src/` | yes | `.py` modules from Phase 3 |
| `/content/data_big/` | **no** | Scratch — regenerate on demand |